In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/Users/adityakinjawadekar/Documents/svm/home/annotated_feature_csvs_all_wavelets/features_patient_001_wavelet_bior3.5.csv')
df = df.dropna(axis=1)
df = df.drop(['d1_hurst'] ,axis =1)

In [4]:
import pandas as pd


df = pd.get_dummies(
    df,
    columns=['channel'],   
    prefix='channel',      
    dtype='int'            
)




In [5]:
df

,label,a4_max,a4_min,a4_mean,a4_std,a4_entropy,a4_fractal_dim,a4_psd,d4_max,d4_min,...,channel_Fp1-F3,channel_Fp1-F7,channel_Fp2-F4,channel_Fp2-F8,channel_Fz-Cz,channel_P4-O2,channel_T3-T5,channel_T4-T6,channel_T5-O1,channel_T6-O2
0,0,-3.729168e-07,-1.546465e-05,-0.000008,0.000004,2.303365,1.256007,1.674155e-13,0.000007,-0.000003,...,0,0,1,0,0,0,0,0,0,0
1,0,1.715476e-05,-7.584807e-06,0.000006,0.000007,1.594403,1.254585,1.428363e-13,0.000003,-0.000003,...,0,0,1,0,0,0,0,0,0,0
2,0,1.542505e-05,-4.443749e-07,0.000009,0.000004,2.453316,1.260100,1.009196e-13,0.000004,-0.000003,...,0,0,1,0,0,0,0,0,0,0
3,0,1.612060e-05,-3.287754e-06,0.000007,0.000004,1.618224,1.255053,2.003186e-13,0.000005,-0.000002,...,0,0,1,0,0,0,0,0,0,0
4,0,1.724258e-05,1.976876e-06,0.000008,0.000005,2.160016,1.259896,1.800631e-13,0.000005,-0.000001,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10518,0,1.995852e-05,-3.858917e-05,-0.000006,0.000021,1.888861,1.282020,2.370834e-12,0.000008,-0.000011,...,0,0,0,0,0,0,0,0,0,0
10519,0,1.083541e-04,-9.481192e-05,-0.000034,0.000043,2.281521,1.320797,2.509055e-11,0.000016,-0.000052,...,0,0,0,0,0,0,0,0,0,0
10520,0,2.342895e-05,-1.424403e-04,-0.000026,0.000049,1.113138,1.284708,8.820790e-12,0.000002,-0.000034,...,0,0,0,0,0,0,0,0,0,0
10521,0,2.960409e-05,1.740585e-05,0.000023,0.000003,2.356738,1.255904,1.126854e-13,0.000003,-0.000004,...,0,0,0,0,0,0,0,0,0,0


In [6]:
annotations = pd.read_excel('/Users/adityakinjawadekar/Documents/svm/home/dataset/anot.xlsx')
annotations = annotations.dropna(axis=0)

In [7]:
import random
eeg_idx = [1, 4, 5, 7, 9, 11, 13, 14, 15, 16, 17, 19, 20, 21, 22, 25, 31, 34,
           36, 38, 39, 40, 41, 44, 47, 50, 52, 62, 63, 66, 67, 69, 73, 75,
           76, 77, 78, 79]

train = int(len(eeg_idx) * 0.75)
test = int(len(eeg_idx) * 0.25)
train_idx = random.sample(eeg_idx,train)
test_idx =[i for i in eeg_idx if i not in train_idx]

In [8]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
BASE_DIR  = Path("/Users/adityakinjawadekar/Documents/svm/home/annotated_feature_csvs_all_wavelets")
WAVELETS  = ['bior3.5', 'coif3', 'db4', 'haar', 'morlet']   # 5 wavelets
# train_idx and test_idx are assumed to be defined exactly as you posted

# ------------------------------------------------------------------
# 1. Prepare containers
# ------------------------------------------------------------------
train_dfs = {w: [] for w in WAVELETS}   # wavelet -> list[pd.DataFrame]
test_dfs  = {w: [] for w in WAVELETS}

# ------------------------------------------------------------------
# 2. Helper to load if the file exists
# ------------------------------------------------------------------
def load_if_exists(pid: int, wavelet: str):
    path = BASE_DIR / f"features_patient_{pid:03d}_wavelet_{wavelet}.csv"
    if path.exists():
        return pd.read_csv(path)
    else:
        print(f"⚠️  Missing file for patient {pid:03d}, wavelet {wavelet}")
        return None

# ------------------------------------------------------------------
# 3. Populate the dictionaries
# ------------------------------------------------------------------
for pid in train_idx:
    for w in WAVELETS:
        df = load_if_exists(pid, w)
        if df is not None:
            train_dfs[w].append(df)

for pid in test_idx:
    for w in WAVELETS:
        df = load_if_exists(pid, w)
        if df is not None:
            test_dfs[w].append(df)

# ------------------------------------------------------------------
# 4.  Usage examples
# ------------------------------------------------------------------
# • train_dfs['db4']   → list of DataFrames (one per patient) in the train split
# • test_dfs['haar']   → list of DataFrames in the test split
#
# To concatenate a wavelet’s list into a single DataFrame:
#   train_db4 = pd.concat(train_dfs['db4'], ignore_index=True)
#   test_db4  = pd.concat(test_dfs['db4'],  ignore_index=True)


In [9]:
db4train = train_dfs['db4']
db4test = test_dfs['db4']

In [10]:
import numpy as np
import pandas as pd
import optuna

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE


def svm_optuna_from_df_lists(
    train_df_list,
    test_df_list,
    label_col: str = "label",
    channel_col: str = "channel",
    n_trials: int = 30,
    k_folds:  int = 3,
    pca_n_components: float | int = 0.95,
    threshold: float | None = None,
    random_state: int = 42,
):
    """
    Train / tune SVM on lists of DataFrames.
    Adds one‑hot encoding for the `channel` column (if present).
    """

    # ------------------------------------------------------------------
    # 1) Concatenate & drop all‑NaN columns / any‑NaN rows
    # ------------------------------------------------------------------
    train_df = pd.concat(train_df_list, ignore_index=True)
    test_df  = pd.concat(test_df_list,  ignore_index=True)

    # Align columns between splits before NaN handling
    common_cols = train_df.columns.intersection(test_df.columns)
    train_df = train_df[common_cols].copy()
    test_df  = test_df[common_cols].copy()

    # Drop columns that are all NaN in train **and** test
    nan_all_cols = [
        c for c in train_df.columns
        if train_df[c].isna().all() and test_df[c].isna().all()
    ]
    train_df.drop(columns=nan_all_cols, inplace=True)
    test_df.drop(columns=nan_all_cols, inplace=True)

    # ------------------------------------------------------------------
    # 2) One‑hot encode the `channel` column (if present)
    # ------------------------------------------------------------------
    if channel_col in train_df.columns:
        train_df = pd.get_dummies(train_df, columns=[channel_col], prefix=channel_col, dtype=int)
        test_df  = pd.get_dummies(test_df,  columns=[channel_col], prefix=channel_col, dtype=int)
        # Re‑align again (test may lack a channel present in train or vice‑versa)
        train_df, test_df = train_df.align(test_df, join="outer", axis=1, fill_value=0)

    # ------------------------------------------------------------------
    # 3) Final NaN drop
    # ------------------------------------------------------------------
    train_df.dropna(axis=0, how="any", inplace=True)
    test_df.dropna(axis=0,  how="any", inplace=True)

    # ------------------------------------------------------------------
    # 4) Split X / y
    # ------------------------------------------------------------------
    X_train_raw = train_df.drop(columns=[label_col]).values
    y_train     = train_df[label_col].values
    X_test_raw  = test_df.drop(columns=[label_col]).values
    y_test      = test_df[label_col].values

    # ------------------------------------------------------------------
    # 5) Scale + PCA
    # ------------------------------------------------------------------
    scaler = MinMaxScaler().fit(X_train_raw)
    X_train_scaled = scaler.transform(X_train_raw)
    X_test_scaled  = scaler.transform(X_test_raw)

    pca = PCA(n_components=pca_n_components, random_state=random_state).fit(X_train_scaled)
    X_train_pca = pca.transform(X_train_scaled)
    X_test_pca  = pca.transform(X_test_scaled)

    # ------------------------------------------------------------------
    # 6) Optuna objective
    # ------------------------------------------------------------------
    def objective(trial):
        C      = trial.suggest_float("C", 1e-3, 1e3, log=True)
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf", "poly", "sigmoid"])
        gamma  = trial.suggest_float("gamma", 1e-4, 10.0, log=True) if kernel != "linear" else "scale"
        degree = trial.suggest_int("degree", 2, 5) if kernel == "poly" else 3

        clf = SVC(
            C=C, kernel=kernel,
            gamma=gamma if kernel != "linear" else "scale",
            degree=degree,
            probability=False,
            random_state=random_state,
        )

        skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=random_state)
        aucs, precisions = [], []

        for tr_idx, val_idx in skf.split(X_train_pca, y_train):
            X_tr, X_val = X_train_pca[tr_idx], X_train_pca[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            X_tr, y_tr = SMOTE(random_state=random_state).fit_resample(X_tr, y_tr)

            clf.fit(X_tr, y_tr)
            scores = clf.decision_function(X_val)
            preds  = clf.predict(X_val)

            aucs.append(roc_auc_score(y_val, scores))
            precisions.append(precision_score(y_val, preds, zero_division=0))

        return 0.2 * np.mean(aucs) + 0.8 * np.mean(precisions)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials,n_jobs=1)

    # ------------------------------------------------------------------
    # 7) Train best model on full, SMOTE‑balanced training data
    # ------------------------------------------------------------------
    bp = study.best_params
    best_clf = SVC(
        C=bp["C"], kernel=bp["kernel"],
        gamma=bp["gamma"] if bp["kernel"] != "linear" else "scale",
        degree=bp["degree"] if bp["kernel"] == "poly" else 3,
        probability=True,
        random_state=random_state,
    )

    X_train_res, y_train_res = SMOTE(random_state=random_state).fit_resample(X_train_pca, y_train)
    best_clf.fit(X_train_res, y_train_res)

    # ------------------------------------------------------------------
    # 8) Evaluate on test set (threshold auto‑tune optional)
    # ------------------------------------------------------------------
    y_probs = best_clf.predict_proba(X_test_pca)[:, 1]

    def metrics_at(t):
        y_pred = (y_probs >= t).astype(int)
        return {
            "Threshold":  t,
            "Accuracy":   accuracy_score(y_test, y_pred),
            "AUROC":      roc_auc_score(y_test, y_probs),
            "F1":         f1_score(y_test, y_pred),
            "Precision":  precision_score(y_test, y_pred, zero_division=0),
            "Recall":     recall_score(y_test, y_pred),
            "Confusion":  confusion_matrix(y_test, y_pred).tolist(),
        }

    if threshold is None:
        best = max((metrics_at(t) for t in np.linspace(0.3, 0.9, 61)), key=lambda m: m["F1"])
    else:
        best = metrics_at(threshold)

    return {
        "model":   best_clf,
        "study":   study,
        "scaler":  scaler,
        "pca":     pca,
        "metrics": best,
    }


In [ ]:
res_db4 = svm_optuna_from_df_lists(
    train_df_list = train_dfs['db4'],
    test_df_list  = test_dfs['db4']
)

print("Best F1 for db4:", res_db4["metrics"]["F1"])


[I 2025-07-03 00:20:47,734] A new study created in memory with name: no-name-499f2fcb-8218-4d52-bf5d-023d90d16eae
